In [1]:
import pandas as pd
import re
from pathlib import Path
from collections import defaultdict

def create_results_dictionary(root_dir_path):
    # Data structure: dict[Algorithm][LLM] = list of dataframes (to be concatenated later)
    temp_storage = defaultdict(lambda: defaultdict(list))
    
    # 1. Iterate over the algorithm folders
    root_path = Path(root_dir_path)
    
    # Assuming the algorithm folders start with "bootstrap_results_" 
    # You can adjust the glob pattern if your folders look different
    for folder in root_path.glob("bootstrap_results_*"):
        
        # Extract Algorithm name (e.g., "G-Greedy" from "bootstrap_results_G-Greedy")
        algo_name = folder.name.replace("bootstrap_results_", "")
        
        # 2. Iterate over parquet files inside the folder
        for file_path in folder.glob("*.parquet"):
            filename = file_path.name
            
            # 3. Parse the filename to get metadata
            # Pattern looks for: sample_{number} AND subject_united_{llm_name}.parquet
            # Example: ...sample_1_subject_united_EleutherAI-pythia-6.9b.parquet
            match = re.search(r"sample_(\d+)_subject_united_(.+)\.parquet", filename)
            
            if match:
                sample_id = int(match.group(1))
                llm_name = match.group(2)
                
                # Read the parquet file
                try:
                    df = pd.read_parquet(file_path)
                    
                    # Add metadata columns so you can distinguish rows after aggregation
                    df['sample_id'] = sample_id
                    # Optional: df['algorithm'] = algo_name
                    
                    # Store in our temp structure
                    temp_storage[algo_name][llm_name].append(df)
                    
                except Exception as e:
                    print(f"Error reading {filename}: {e}")

    # 4. Final Aggregation: Concatenate lists of DFs into single DFs
    final_results = {}
    
    for algo, llm_dict in temp_storage.items():
        final_results[algo] = {}
        for llm, df_list in llm_dict.items():
            if df_list:
                # Concatenate all samples for this specific Algo/LLM combo
                final_results[algo][llm] = pd.concat(df_list, ignore_index=True)

    return final_results

# --- Usage ---
# Replace '.' with the path to the folder containing your 3 algorithm folders
results = create_results_dictionary('.')

# Example check
# print(results['G-Greedy']['EleutherAI-pythia-6.9b'].head())

In [ ]:
LLMS = [
"EleutherAI-pythia-6.9b",
"meta-llama-Llama-3.1-8B",
"meta-llama-Meta-Llama-3-8B-Instruct"
]
ALGORITHMS = [
"G-Greedy",
"G-Utility",
"Random",
]
Player_G_Utility = 'Perplexity'
Player_F_Utility = 'NormalizedViewCount'

NAME_MAP = {
    'G-Utility': 'G-Utility',
    'Random_player_f_classifier': 'Random',
    'G-Greedy_classifier_f': 'G-Greedy',
}

In [4]:
NAME_MAP = {
    'G-Utility': 'G-Utility',
    'Random_player_f_classifier': 'Random',
    'G-Greedy_classifier_f': 'G-Greedy',
}

# .get(k, k) ensures that if a key isn't in NAME_MAP, it just keeps its original name.
new_results = {NAME_MAP.get(k, k): v for k, v in results.items()}

# If you want to overwrite the original variable:
results = new_results

In [ ]:
from scipy import stats
from itertools import combinations
for llm in LLMS:
    series={}
    print(f"\n{'='*20} Evaluating LLM: {llm} {'='*20}")
    for algo in ALGORITHMS:
            results[algo][llm]["relevant_score"] = results[algo][llm][Player_G_Utility] * results[algo][llm]["genai_label"]
            series[algo] = results[algo][llm].groupby('sample_id')['relevant_score'].sum()
            pd.DataFrame(series[algo]).to_parquet(f"strategy_total_results/series_{algo}_{Player_G_Utility}_{llm}.pq")
    # --- 3. Run Pairwise Comparisons ---
    
    # Helper function to run and print paired t-test
    def run_paired_test(name_a, series_a, name_b, series_b):
        # Align data for this specific pair
        a_aligned, b_aligned = series_a.align(series_b, join='inner')
        
        # Paired T-Test
        t_stat, p_val = stats.ttest_rel(a_aligned, b_aligned)
        diff_mean = a_aligned.mean() - b_aligned.mean()
        
        print(f"\n--- Comparison: {name_a} vs {name_b} ---")
        print(f"Difference in Means ({name_a} - {name_b}): {diff_mean:.4f}")
        print(f"Mean {name_a}: {a_aligned.mean():.4f}")
        print(f"Mean {name_b}: {b_aligned.mean():.4f}")
        print(f"T-statistic: {t_stat:.4f}")
        print(f"P-value: {p_val:.4e}")
        if p_val < 0.01:
            winner = name_a if diff_mean > 0 else name_b
            print(f"Result: Significant difference (Winner: {winner})")
        else:
            print("Result: No significant difference")
    for alg1,alg2 in combinations(ALGORITHMS, 2):
        run_paired_test(alg1, series[alg1], alg2, series[alg2])


In [ ]:
from scipy import stats
from itertools import combinations
# Helper function to run and print paired t-test
def run_paired_test(name_a, series_a, name_b, series_b):
        # Align data for this specific pair
        a_aligned, b_aligned = series_a.align(series_b, join='inner')
        
        # Paired T-Test
        t_stat, p_val = stats.ttest_rel(a_aligned, b_aligned)
        diff_mean = a_aligned.mean() - b_aligned.mean()
        
        print(f"\n--- Comparison: {name_a} vs {name_b} ---")
        print(f"Difference in Means ({name_a} - {name_b}): {diff_mean:.4f}")
        print(f"Mean {name_a}: {a_aligned.mean():.4f}")
        print(f"Mean {name_b}: {b_aligned.mean():.4f}")
        print(f"T-statistic: {t_stat:.4f}")
        print(f"P-value: {p_val:.4e}")
        if p_val < 0.01:
            winner = name_a if diff_mean > 0 else name_b
            print(f"Result: Significant difference (Winner: {winner})")
        else:
            print("Result: No significant difference")
for llm in LLMS:
    series={}
    print(f"\n{'='*20} Evaluating LLM: {llm} {'='*20}")
    for algo in ALGORITHMS:
            results[algo][llm]["relevant_score"] = results[algo][llm][Player_F_Utility] * results[algo][llm]["forum_label"].astype(int)
            series[algo] = results[algo][llm].groupby('sample_id')['relevant_score'].sum()
            pd.DataFrame(series[algo]).to_parquet(f"strategy_total_results/series_{algo}_{Player_F_Utility}_{llm}.pq")

    # --- 3. Run Pairwise Comparisons ---
    
    
    for alg1,alg2 in combinations(ALGORITHMS, 2):
        run_paired_test(alg1, series[alg1], alg2, series[alg2])
